# Extensions: system prompt, fixes, spelling variation, more models, hidden states

Follow-up experiments for the Egyptian Arabic / Arabizi consistency benchmark. Run on a Colab GPU (T4 is enough for Qwen3-4B).

Every experiment saves a CSV in `outputs/` and can be resumed: if Colab disconnects, run the cell again and it continues from where it stopped. Sections 1 to 5 produce model answers that still need scoring with the same rubric as the main benchmark. Section 6 needs no scoring.

| Section | Question | Prompts |
|---|---|---|
| 1 | Does the Arabizi gap survive when the model knows it is a support assistant? | 400 |
| 2 | Does a one-sentence hint about Arabizi help? | 100 |
| 3 | Does letting the model rewrite Arabizi in Arabic script first help? | 100 (two passes) |
| 4 | How much do scores move when the same Arabizi request is spelled differently? | checked rows of the variants sheet |
| 5 | Does the gap shrink with model size, or with Arabic-focused training? | 500 per model |
| 6 | Where inside the model do Arabizi and Arabic script stop looking like the same request? | no generation |

In [ ]:
!pip -q install bitsandbytes
!test -d egyptian-arabizi-consistency || git clone -q https://github.com/nour-rayann/egyptian-arabizi-consistency.git

Connects Google Drive and saves every answer there. Approve the pop-up. It should print `Saving to: /content/drive/MyDrive/arabizi_outputs`.

In [ ]:
# Every answer is saved to Google Drive straight away, so nothing is lost if Colab disconnects.
# Run this cell first in every session: it shows what is already saved, and later cells continue from there.
import os
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/arabizi_outputs")
OUT.mkdir(parents=True, exist_ok=True)
assert os.path.ismount("/content/drive"), "Google Drive is not connected. Run this cell again and approve the pop-up."

def save_csv(df, path):
    """Write the file completely before replacing the old one, so a disconnect can never leave half a file."""
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        df.to_csv(f, index=False)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    try:
        os.replace(tmp, path)
    except OSError:                      # if Drive refuses the swap, write the file directly
        df.to_csv(path, index=False)
    if not Path(path).exists():
        raise RuntimeError(f"Could not save {path} to Google Drive. Stop and check the Drive connection.")

def load_saved(path):
    """Read what is already saved. Stops with a clear message instead of silently starting over."""
    if not Path(path).exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        raise RuntimeError(f"{path} exists on Drive but cannot be read ({e}). Do NOT delete it; ask before continuing.")

def show_progress():
    files = sorted(OUT.glob("*.csv"))
    print("Saving to:", OUT)
    if not files:
        print("Nothing saved yet.")
    for f in files:
        print(f"  {f.name:45s} {len(load_saved(f)):4d} answers saved")

show_progress()

In [ ]:
import gc, difflib
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

REPO = Path("egyptian-arabizi-consistency")
bench = pd.read_csv(REPO / "data" / "results_400.csv")          # prompts, gold labels, baseline answers and scores
q1 = pd.read_csv(REPO / "data" / "q1_script_test_results.csv")  # Arabizi requests rewritten in Arabic script
arabizi = bench[bench.condition == "Arabizi + English"].reset_index(drop=True)
GOLD = ["gold_intent", "expected_action", "forbidden_assumption"]

MAIN_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
model = tokenizer = None

def load_model(name, four_bit=False):
    """Load a model, freeing the previous one first. four_bit=True fits 7-8B models on a T4."""
    global model, tokenizer
    model = tokenizer = None
    gc.collect(); torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(name)
    kwargs = dict(device_map="auto")
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kwargs["torch_dtype"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    model.eval()
    print("loaded", name)

def chat_text(prompt, system=None):
    """Same chat formatting as the main benchmark. enable_thinking=False only matters for Qwen3 thinking models."""
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        # some chat templates reject a system role: put the system text at the top of the user message
        messages = [{"role": "user", "content": (system + "\n\n" if system else "") + prompt}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def ask(prompt, system=None, max_new_tokens=300):
    inputs = tokenizer([chat_text(prompt, system)], return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def run(rows, prompt_col, out_name, system=None, id_col="example_id", keep=()):
    """Answer every row and save to outputs/<out_name>.csv after each answer. Safe to re-run."""
    path = OUT / f"{out_name}.csv"
    if not str(path).startswith("/content/drive/") or not os.path.ismount("/content/drive"):
        raise RuntimeError(f"Not saving to Google Drive ({path}). Run cell 2 first, then this cell again.")
    done = load_saved(path)
    print(f"{out_name}: {len(done)} of {len(rows)} already saved in {path}")
    first_new = True
    seen = set(done[id_col]) if len(done) else set()
    records = done.to_dict("records")
    for _, r in tqdm(rows.iterrows(), total=len(rows), desc=out_name):
        if r[id_col] in seen:
            continue
        rec = {id_col: r[id_col], "prompt": r[prompt_col], "model_response": ask(r[prompt_col], system)}
        rec.update({k: r[k] for k in keep})
        records.append(rec)
        save_csv(pd.DataFrame(records), path)
        if first_new:
            print(f"First new answer saved to Drive: {path.name} now has {len(load_saved(path))} rows")
            first_new = False
    return load_saved(path)

In [ ]:
MAIN_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
load_model(MAIN_MODEL)

# Sanity check: greedy decoding should reproduce the baseline answer for E001
print(ask(bench.prompt[0])[:200])
print("matches baseline:", ask(bench.prompt[0]).strip() == str(bench.model_response[0]).strip())

## 1. System prompt

The main benchmark used no system prompt, so the model did not know it was a support assistant. Here it is told. Compare with the baseline per condition.

In [ ]:
SYSTEM = "You are a customer support assistant for an online store. Help the customer with their request."
sys_all = run(bench, "prompt", "sysprompt_400", system=SYSTEM, keep=("scenario_id", "condition", "scope"))

## 2. Prompt hint

Same system prompt plus one sentence telling the model about Arabizi. Compare with the Arabizi rows of section 1, so the only difference is the hint.

In [ ]:
HINT = (SYSTEM + " Customers may write in English, in Egyptian Arabic, or in Egyptian Arabic typed in Latin letters "
        "and numbers (Arabizi, where 3 = ع, 7 = ح, 2 = ء, 5 = خ, 8 = غ), often mixed with English words.")
hint_arabizi = run(arabizi, "prompt", "hint_arabizi_100", system=HINT, keep=("scenario_id", "scope"))

## 3. Transliterate first

Two passes. First the model rewrites the Arabizi message in Arabic script, then it answers the rewritten message (with the section 1 system prompt). The hand rewrite in `q1_transliteration_sheet.csv` showed this works when a person does the rewriting; this tests whether the model can do it on its own.

The first pass can be checked automatically against the hand rewrite, with no scoring needed.

In [ ]:
REWRITE = ("Rewrite this message in Arabic script. It is Egyptian Arabic typed in Latin letters and numbers. "
           "Keep English words exactly as they are. Reply with the rewritten message only.\n\n")

path = OUT / "translit_first_100.csv"
if not str(path).startswith("/content/drive/"):
    raise RuntimeError("Not saving to Google Drive. Run cell 2 first.")
done = load_saved(path)
print(f"translit_first_100: {len(done)} of {len(arabizi)} already saved in {path}")
seen = set(done.example_id) if len(done) else set()
records = done.to_dict("records")
hand = q1.set_index("arabizi_example_id").arabic_script_exact
for _, r in tqdm(arabizi.iterrows(), total=len(arabizi), desc="translit_first"):
    if r.example_id in seen:
        continue
    rewritten = ask(REWRITE + r.prompt, max_new_tokens=100).strip()
    records.append(dict(example_id=r.example_id, scenario_id=r.scenario_id, scope=r["scope"], prompt=r.prompt,
                        model_rewrite=rewritten, hand_rewrite=hand[r.example_id],
                        model_response=ask(rewritten, system=SYSTEM)))
    save_csv(pd.DataFrame(records), path)
tf = load_saved(path)

tf["rewrite_similarity"] = [difflib.SequenceMatcher(None, str(a), str(b)).ratio() for a, b in zip(tf.model_rewrite, tf.hand_rewrite)]
save_csv(tf, path)
print(f"Model rewrite vs hand rewrite: mean character similarity {tf.rewrite_similarity.mean():.2f}, "
      f"identical in {(tf.model_rewrite.str.strip() == tf.hand_rewrite.str.strip()).sum()} of {len(tf)}")
tf.sort_values("rewrite_similarity")[["prompt", "model_rewrite", "hand_rewrite"]].head(10)

## 4. Spelling variation

Upload the checked variants sheet (`arabizi_spelling_variants_draft.csv`, rows with `checked = 1`) to the Colab file panel first. No system prompt here, so answers are comparable with the baseline Arabizi answers.

In [ ]:
variants = pd.read_csv("arabizi_spelling_variants_draft.csv")
variants = variants[variants.checked == 1].reset_index(drop=True)
variants["variant_id"] = variants.arabizi_example_id + "_" + variants.variant_type
print(len(variants), "checked variants")
spell = run(variants, "variant", "spelling_variants", id_col="variant_id",
            keep=("scenario_id", "arabizi_example_id", "scope", "variant_type", "original"))

## 5. More models

In-scope English, Arabizi and the Arabic-script rewrite (255 prompts per model), same settings as the main benchmark. Run one model at a time. ALLaM is gated: accept its licence on Hugging Face and run `from huggingface_hub import login; login()` once first. The 7-8B models are loaded in 4-bit to fit a T4.

In [ ]:
# Needed once per session for ALLaM (gated model). Paste your Hugging Face token when asked.
from huggingface_hub import login
login()

In [ ]:
# In-scope English, Arabizi and the Arabic-script rewrite only: 255 prompts per model.
# Finished models are skipped. If Colab stops, run cell 2, cell 3, the login cell and this cell again:
# it continues from the last saved answer.
MODELS = [
    ("Qwen/Qwen3-1.7B", False),
    ("Qwen/Qwen3-8B", True),
    ("ALLaM-AI/ALLaM-7B-Instruct-preview", True),
]
ins = bench[(bench.scope == "in_scope") & bench.condition.isin(["English", "Arabizi + English"])]
rewrites = q1[q1.scope == "in_scope"].rename(columns={"arabizi_example_id": "example_id"})
print(len(ins), "benchmark prompts +", len(rewrites), "rewrites per model")   # 170 + 85

for name, four_bit in MODELS:
    tag = name.split("/")[-1]
    saved = len(load_saved(OUT / f"model_{tag}_400.csv")), len(load_saved(OUT / f"model_{tag}_script_100.csv"))
    if saved == (len(ins), len(rewrites)):
        print(f"{tag}: already finished on Drive ({saved[0]} + {saved[1]}), skipping")
        continue
    print(f"{tag}: {saved[0]} of {len(ins)} and {saved[1]} of {len(rewrites)} saved so far, continuing")
    load_model(name, four_bit)
    run(ins, "prompt", f"model_{tag}_400", keep=("scenario_id", "condition", "scope"))
    run(rewrites, "arabic_script_exact", f"model_{tag}_script_100", keep=("scenario_id", "scope"))
show_progress()

## 6. Hidden states: where do Arabizi and Arabic script diverge?

For every prompt, take the model's internal vector at the last position of the input (the point where it starts writing its answer), at every layer. If the model treats an Arabizi request and its Arabic-script twin as the same request, their vectors should be close.

Measured per layer with a retrieval test: for each Arabizi request, is its own Arabic-script twin the nearest of the 100 Arabic-script requests? Chance is 1%. Two reference pairs are shown for comparison: English against Arabic + English, and Arabizi against English.

Vectors are centred per layer before comparing, because raw hidden states all point in a similar direction.

In [ ]:
load_model(MAIN_MODEL)   # full precision, same as the main benchmark

@torch.no_grad()
def last_token_states(prompts):
    """Array of shape (prompts, layers + 1, hidden size)."""
    out = []
    for p in tqdm(prompts):
        inputs = tokenizer([chat_text(p)], return_tensors="pt", add_special_tokens=False).to(model.device)
        hs = model(**inputs, output_hidden_states=True).hidden_states
        out.append(torch.stack([h[0, -1] for h in hs]).float().cpu().numpy())
    return np.stack(out)

by_cond = {c: g.sort_values("scenario_id") for c, g in bench.groupby("condition")}
q1s = q1.sort_values("scenario_id")
sets = {
    "English": by_cond["English"].prompt.tolist(),
    "Arabic + English": by_cond["Egyptian Arabic + English"].prompt.tolist(),
    "Arabizi": by_cond["Arabizi + English"].prompt.tolist(),
    "Arabizi in Arabic script": q1s.arabic_script_exact.tolist(),
}
states = {k: last_token_states(v) for k, v in sets.items()}
np.savez_compressed(OUT / "hidden_states.npz", **states)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

n_layers = states["English"].shape[1]
mean_all = np.concatenate(list(states.values())).mean(0)          # per-layer centre over all prompts

def unit(name, layer):
    x = states[name][:, layer] - mean_all[layer]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def compare(a, b):
    """Per layer: top-1 retrieval accuracy, mean similarity of true pairs, mean similarity of wrong pairs."""
    rows = []
    for L in range(n_layers):
        sim = unit(a, L) @ unit(b, L).T
        n = len(sim)
        rows.append(dict(layer=L, retrieval=(sim.argmax(1) == np.arange(n)).mean(),
                         matched=np.diag(sim).mean(), mismatched=(sim.sum() - np.trace(sim)) / (n * n - n)))
    return pd.DataFrame(rows)

PAIRS = [("Arabizi", "Arabizi in Arabic script"), ("English", "Arabic + English"), ("Arabizi", "English")]
res = {f"{a} vs {b}": compare(a, b) for a, b in PAIRS}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for label, r in res.items():
    ax[0].plot(r.layer, r.retrieval, label=label)
    ax[1].plot(r.layer, r.matched - r.mismatched, label=label)
ax[0].set(title="Is the twin the nearest request?", xlabel="Layer", ylabel="Top-1 retrieval accuracy")
ax[1].set(title="Similarity of true pairs minus wrong pairs", xlabel="Layer")
ax[0].legend()
plt.tight_layout(); plt.savefig(OUT / "hidden_state_similarity.png", dpi=150); plt.show()
pd.concat(res, names=["pair"]).reset_index(level=0).to_csv(OUT / "hidden_state_similarity.csv", index=False)

Does internal similarity predict behaviour? For each scenario, compare how close the Arabizi request is to its Arabic-script twin with the intent score the Arabizi answer received in the main benchmark.

In [ ]:
key = "Arabizi vs Arabizi in Arabic script"
best = int(res[key].retrieval.idxmax())
pair_sim = (unit("Arabizi", best) * unit("Arabizi in Arabic script", best)).sum(1)
scores = by_cond["Arabizi + English"].intent_score.values
rho, p = spearmanr(pair_sim, scores)
print(f"Layer {best} (best retrieval layer): Spearman correlation between twin similarity and Arabizi intent score = {rho:.2f} (p = {p:.2g})")
print(pd.DataFrame({"intent_score": scores, "twin_similarity": pair_sim}).groupby("intent_score").twin_similarity.agg(["count", "mean"]).round(3))

### Control: is the link about meaning, or about shared English words?

An Arabizi request and its Arabic-script rewrite contain the same English terms in Latin letters, which could explain why they look alike inside the model. Pure Egyptian Arabic shares no Latin-letter words with Arabizi. This cell repeats the test against that condition. It stands on its own: it needs cells 1, 2 and 3, not the earlier section 6 cells.

In [ ]:
# Hidden states, control test: is the Arabizi link about meaning, or about shared English words?
# Pure Egyptian Arabic shares no Latin-letter English words with Arabizi. If the twin is still found
# well above chance (1%), the model links the Egyptian words themselves.
# Needs cells 1, 2 and 3 only. Takes about 5-10 minutes. Saves everything to Drive.
import numpy as np, torch, matplotlib.pyplot as plt
from scipy.stats import spearmanr

if model is None or getattr(model.config, "_name_or_path", "") != MAIN_MODEL:
    load_model(MAIN_MODEL)      # full precision, same as the main benchmark

@torch.no_grad()
def last_token_states(prompts):
    out = []
    for p in tqdm(prompts):
        inputs = tokenizer([chat_text(p)], return_tensors="pt", add_special_tokens=False).to(model.device)
        hs = model(**inputs, output_hidden_states=True).hidden_states
        out.append(torch.stack([h[0, -1] for h in hs]).float().cpu().numpy())
    return np.stack(out)

by_cond = {c: g.sort_values("scenario_id") for c, g in bench.groupby("condition")}
q1s = q1.sort_values("scenario_id")
sets = {
    "Arabizi": by_cond["Arabizi + English"].prompt.tolist(),
    "Arabizi in Arabic script": q1s.arabic_script_exact.tolist(),
    "Egyptian Arabic": by_cond["Egyptian Arabic"].prompt.tolist(),
    "English": by_cond["English"].prompt.tolist(),
}
assert all((g.scenario_id.values == q1s.scenario_id.values).all() for g in by_cond.values()), "scenarios not aligned"

path = OUT / "hidden_states_control.npz"
if path.exists():
    states = dict(np.load(path)); print("Loaded saved hidden states from Drive")
else:
    states = {k: last_token_states(v) for k, v in sets.items()}
    np.savez_compressed(path, **states); print("Saved hidden states to", path)

n_layers = states["English"].shape[1]
mean_all = np.concatenate(list(states.values())).mean(0)

def unit(name, layer):
    x = states[name][:, layer] - mean_all[layer]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def compare(a, b):
    rows = []
    for L in range(1, n_layers):      # layer 0 is the same token for every prompt, so it is skipped
        sim = unit(a, L) @ unit(b, L).T
        n = len(sim)
        rows.append(dict(layer=L, retrieval=(sim.argmax(1) == np.arange(n)).mean(),
                         gap=np.diag(sim).mean() - (sim.sum() - np.trace(sim)) / (n * n - n)))
    return pd.DataFrame(rows)

PAIRS = [("Arabizi", "Arabizi in Arabic script"),   # shares English words: the original test
         ("Arabizi", "Egyptian Arabic"),            # the control: no shared Latin-letter words
         ("Arabizi in Arabic script", "Egyptian Arabic"),  # same script, slightly different wording
         ("English", "Egyptian Arabic")]            # reference: a pair the model understands
res = {f"{a} vs {b}": compare(a, b) for a, b in PAIRS}

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for label, r in res.items():
    ax[0].plot(r.layer, r.retrieval, label=label)
    ax[1].plot(r.layer, r.gap, label=label)
ax[0].axhline(0.01, color="grey", ls=":", lw=1)
ax[0].set(title="Is the twin the nearest request? (chance = 1%)", xlabel="Layer", ylabel="Top-1 retrieval accuracy")
ax[1].set(title="Similarity of true pairs minus wrong pairs", xlabel="Layer")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "hidden_state_control.png", dpi=150); plt.show()
pd.concat(res, names=["pair"]).reset_index(level=0).to_csv(OUT / "hidden_state_control.csv", index=False)

print("\nBest layer per pair (retrieval):")
for label, r in res.items():
    b = r.loc[r.retrieval.idxmax()]
    print(f"  {label:40s} {b.retrieval:.0%} at layer {int(b.layer)}   (last layer: {r.retrieval.iloc[-1]:.0%})")

key = "Arabizi vs Egyptian Arabic"
best = int(res[key].loc[res[key].retrieval.idxmax(), "layer"])
pair_sim = (unit("Arabizi", best) * unit("Egyptian Arabic", best)).sum(1)
scores = by_cond["Arabizi + English"].intent_score.values
rho, p = spearmanr(pair_sim, scores)
print(f"\n{key}, layer {best}: Spearman correlation with Arabizi intent score = {rho:.2f} (p = {p:.2g})")
print(pd.DataFrame({"intent_score": scores, "twin_similarity": pair_sim}).groupby("intent_score").twin_similarity.agg(["count", "mean"]).round(3))
print("\nSaved to Drive: hidden_states_control.npz, hidden_state_control.png, hidden_state_control.csv")

## 7. Scoring sheet

Collects every new answer into one sheet with the gold labels and empty score columns, ready for the same rubric as the main benchmark (intent 0-2, action 0-2, unsupported action 0/1).

In [ ]:
gold = bench.drop_duplicates("scenario_id").set_index("scenario_id")[GOLD]
frames = []
for f in sorted(OUT.glob("*.csv")):
    if f.stem in ("to_score", "scored"):      # skip the scoring sheets themselves
        continue
    d = pd.read_csv(f)
    if "model_response" not in d.columns:
        continue
    d.insert(0, "experiment", f.stem)
    d["row_id"] = d[[c for c in ("variant_id", "example_id") if c in d.columns][0]]
    frames.append(d.join(gold, on="scenario_id"))
sheet = pd.concat(frames, ignore_index=True)
cols = ["experiment", "row_id", "scenario_id", "scope", "condition", "prompt", "model_rewrite"] + GOLD + ["model_response"]
sheet = sheet.reindex(columns=cols)
for c in ["intent_score", "action_score", "unsupported_action", "response_type"]:
    sheet[c] = ""
sheet.to_csv(OUT / "to_score.csv", index=False, encoding="utf-8-sig")
print(sheet.experiment.value_counts())

After scoring, save the sheet as `outputs/scored.csv` and run this to get the in-scope means per experiment.

In [ ]:
s = pd.read_csv(OUT / "scored.csv")
s = s[s["scope"] == "in_scope"]
s["condition"] = s.condition.fillna("Arabizi + English")
s.loc[s.experiment.str.endswith("script_100"), "condition"] = "Arabizi rewritten in Arabic script"
print(s.groupby(["experiment", "condition"])[["intent_score", "action_score"]].agg(["mean", "count"]).round(2))